In [ ]:
import logging
from datetime import datetime
from pyspark.sql import DataFrame as SparkDataFrame
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, when

logger = logging.getLogger("churn_model")


def _bucket_age(df: SparkDataFrame) -> SparkDataFrame:
    """Bucket age into groups."""
    return df.withColumn(
        "age_group",
        when(col("age") < 25, "young")
        .when(col("age") < 45, "middle")
        .when(col("age") < 60, "senior")
        .otherwise("elderly")
    )


def _bucket_gender(df: SparkDataFrame) -> SparkDataFrame:
    """Bucket gender into groups."""
    return df.withColumn(
        "gender_group",
        when(col("gender").isNull(), "unknown")
        .when(col("gender").isin("M", "Male"), "male")
        .when(col("gender").isin("F", "Female"), "female")
        .otherwise("other")
    )


def _bucket_location(df: SparkDataFrame) -> SparkDataFrame:
    """Bucket location into groups."""
    return df.withColumn(
        "location_group",
        when(col("location").isNull(), "unknown")
        .when(col("location").isin("NSW", "VIC", "QLD"), "high_density")
        .when(col("location").isin("WA", "SA", "TAS"), "medium_density")
        .otherwise("low_density")
    )


def _engineer_demographic_features(df: SparkDataFrame) -> SparkDataFrame:
    """Engineer demographic-based features using feature_groups."""
    demographic_cols = config.feature_groups.demographic  # ["age", "gender", "location"]
    
    for feature in demographic_cols:
        if feature not in df.columns:
            continue
        
        if feature == "age":
            df = _bucket_age(df)
        elif feature == "gender":
            df = _bucket_gender(df)
        elif feature == "location":
            df = _bucket_location(df)
    
    return df


def _engineer_behaviour_features(df: SparkDataFrame) -> SparkDataFrame:
    """Engineer behaviour-based features using feature_groups."""
    behaviour_cols = config.feature_groups.behaviour  # ["logins", "session_length", "purchase_count"]
    
    # Create engagement score from logins and session_length
    if "logins" in behaviour_cols and "logins" in df.columns:
        if "session_length" in df.columns:
            df = df.withColumn(
                "engagement_score",
                col("logins") * col("session_length") / 100
            )
    
    # Create repeat buyer flag from purchase_count
    if "purchase_count" in behaviour_cols and "purchase_count" in df.columns:
        df = df.withColumn(
            "is_repeat_buyer",
            (col("purchase_count") > 1).cast("double")
        )
    
    return df


def _engineer_financial_features(df: SparkDataFrame) -> SparkDataFrame:
    """Engineer financial-based features using feature_groups."""
    financial_cols = config.feature_groups.financial  # ["monthly_spend", "credit_score", "payment_history"]
    
    # Create spend to credit ratio
    if "monthly_spend" in financial_cols and "monthly_spend" in df.columns:
        if "credit_score" in df.columns:
            df = df.withColumn(
                "spend_to_credit_ratio",
                col("monthly_spend") / (col("credit_score") + 1)
            )
    
    # Create payment issues flag
    if "payment_history" in financial_cols and "payment_history" in df.columns:
        df = df.withColumn(
            "has_payment_issues",
            (col("payment_history") < 0.7).cast("double")
        )
    
    return df


def engineer_features(df: SparkDataFrame) -> SparkDataFrame:
    """Uses feature_groups to engineer features for the churn model."""
    logger.info(f"engineer_features started at {datetime.now()}")
    
    # Step 1: Create tenure if not present
    if "tenure" not in df.columns:
        df = df.withColumn(
            "tenure",
            F.round(F.months_between(F.current_date(), col("customer_since"))).cast("double"),
        )
    
    # Step 2: Engineer demographic features
    df = _engineer_demographic_features(df)
    
    # Step 3: Engineer behaviour features
    df = _engineer_behaviour_features(df)
    
    # Step 4: Engineer financial features
    df = _engineer_financial_features(df)

    logger.info(f"engineer_features finished at {datetime.now()}")
    return df